In [2]:
!pip install -q deepctr-torch


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, which is not installed.
keras-hub 0.21.1 requires tensorflow-text; platform_system != "Windows", which is not installed.


In [3]:
!pip uninstall -y dopamine-rl keras-hub tensorflow-text tf-keras


Found existing installation: dopamine_rl 4.1.2
Uninstalling dopamine_rl-4.1.2:
  Successfully uninstalled dopamine_rl-4.1.2
Found existing installation: keras-hub 0.21.1
Uninstalling keras-hub-0.21.1:
  Successfully uninstalled keras-hub-0.21.1


In [4]:
from deepctr_torch.inputs import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM, DIN
print("deepctr-torch imports OK")


deepctr-torch imports OK


In [7]:
import torch, deepctr_torch, pandas as pd, numpy as np


In [8]:
print("torch:", torch.__version__, "deepctr-torch:", deepctr_torch.__version__)


torch: 2.8.0+cu126 deepctr-torch: 0.2.9


In [30]:
from deepctr_torch.models import DeepFM, DIN
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler , OrdinalEncoder
from deepctr_torch.inputs import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names

In [20]:
from google.colab import drive
drive.mount('/content/drive')
train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
df = train.copy()

In [22]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )

In [25]:
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
df = df[~df['inventory_id'].isin([92, 21])]

In [24]:
label_col, seq_col = 'clicked', 'seq'

In [26]:
# 범주 - 명목
nominal_sparse = ['gender','age_group','inventory_id','day_of_week','hour']

# 범주 - 서열
ordinal_sparse = ['l_feat_3','l_feat_27','feat_e_4','feat_a_1','feat_a_3','feat_a_4','feat_a_8','feat_a_13','feat_a_16','feat_a_18']

# 연속형
all_cols = df.columns.tolist(); base_exclude = set([label_col, seq_col]) | set(ordinal_sparse) | set(nominal_sparse); dense_features = [c for c in all_cols if c not in base_exclude]



In [27]:
# 혹시나 결측치 처리
df[ordinal_sparse + nominal_sparse] = df[ordinal_sparse + nominal_sparse].fillna("-1").astype(str)
df[dense_features] = df[dense_features].replace([np.inf, -np.inf], np.nan).fillna(0.0)


In [29]:
# 명목형은 LabelEncoder
for f in nominal_sparse: df[f] = LabelEncoder().fit_transform(df[f])

# 순서형은 OrdinalEncoder
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1); df[ordinal_sparse] = ord_enc.fit_transform(df[ordinal_sparse])
df[[f"{c}_ordscore" for c in ordinal_sparse]] = MinMaxScaler().fit_transform(df[ordinal_sparse])

# 연속형을 0~1로 스케일
df[dense_features] = MinMaxScaler().fit_transform(df[dense_features])



/tmp/ipython-input-4229765275.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[[f"{c}_ordscore" for c in ordinal_sparse]] = MinMaxScaler().fit_transform(df[ordinal_sparse])
/tmp/ipython-input-4229765275.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[[f"{c}_ordscore" for c in ordinal_sparse]] = MinMaxScaler().fit_transform(df[ordinal_sparse])
/tmp/ipython-input-4229765275.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform